# Exploration des données SIRENE

Ici on regarde le répertoire SIRENE de l'Insee. C'est en gros la liste de **toutes les entreprises de France** (les « unités légales ») avec leur **siège**.

⚠️ **Attention à la RAM** : il y a 30 millions de lignes, donc si on charge tout avec `pd.read_parquet` le kernel plante. On passe toujours par les fonctions de `scripts/data_utils.py` : elles ne gardent que les colonnes demandées et compressent le texte. Après une étape lourde, on fait `del` puis `free_memory()`.

### Lexique : SIRENE

| Terme | En clair |
|---|---|
| **Unité légale (UL)** | L'entreprise au sens juridique. Ça peut être une société (SARL, SAS…) ou une personne qui travaille à son compte (entrepreneur individuel). Elle a un numéro **SIREN** à 9 chiffres. |
| **Établissement** | Un lieu où l'entreprise travaille : un magasin, une usine, un bureau. Une entreprise peut en avoir plusieurs. Chacun a un **SIRET** = SIREN + **NIC** (5 chiffres). |
| **Siège** | L'établissement qui sert d'adresse officielle à l'entreprise. |
| **État administratif** | Pour l'entreprise : `A` = active, `C` = cessée. Pour l'établissement : `A` = ouvert, `F` = fermé. |
| **`dateDebut`** | La date où la situation actuelle de l'entreprise a commencé. Si l'entreprise est cessée, c'est en pratique sa date de cessation. |
| **Statut de diffusion** | `O` = tout est public. `P` = diffusion partielle : la personne a demandé à cacher certaines infos (son sexe apparaît alors en `[ND]`, son adresse est masquée…). |
| **Unité purgée** | `true` = une entreprise fermée depuis longtemps, dont l'Insee a effacé une partie des infos. |
| **Tranche d'effectifs** | Le nombre de salariés, par tranche : `NN` = pas d'info, `00` = 0 salarié, `01` = 1 à 2, `02` = 3 à 5, `03` = 6 à 9, `11` = 10 à 19, `12` = 20 à 49, `21` = 50 à 99, `22` = 100 à 199, `31` = 200 à 249, `32` = 250 à 499, `41` = 500 à 999, `42` = 1 000 à 1 999, `51` = 2 000 à 4 999, `52` = 5 000 à 9 999, `53` = 10 000 et plus. |
| **Catégorie d'entreprise** | La taille calculée par l'Insee : **PME** (petite et moyenne entreprise), **ETI** (taille intermédiaire), **GE** (grande entreprise). Vide quand l'Insee ne l'a pas calculée. |
| **Catégorie juridique** | La forme juridique, sur 4 chiffres. Le premier chiffre donne la grande famille : `1` = entrepreneur individuel, `5` = société commerciale (SARL, SAS, SA…), `6` = autres sociétés privées (SCI, GAEC…), `9` = associations, `7` = organismes publics, etc. |
| **Code NAF (ou APE)** | Le code du secteur d'activité, par exemple `47.11F` pour les hypermarchés. Les 2 premiers chiffres donnent la grande branche (la « division »). Le code a changé plusieurs fois : NAP (avant 1993), NAF1993, NAFRev1, **NAFRev2** (depuis 2008) et **NAF25** (la toute dernière version). |

## 1. Chargement

In [1]:
import sys
sys.path.append("../scripts")

import pandas as pd
from pathlib import Path
from data_utils import load_sirene_ul, load_sirene_sieges, free_memory, mem

pd.set_option("display.max_rows", 60)


def vide(s, label="(vide)"):
    # Renomme la catégorie vide "" d'une colonne category (pour l'affichage)
    return s.cat.rename_categories(lambda x: x or label)


print(mem())

process : 0.14 Go | disponible : 11.26 Go


In [2]:
UL_COLS = [
    "siren", "statutDiffusionUniteLegale", "unitePurgeeUniteLegale",
    "dateCreationUniteLegale", "dateDebut", "etatAdministratifUniteLegale",
    "trancheEffectifsUniteLegale", "anneeEffectifsUniteLegale",
    "categorieEntreprise", "categorieJuridiqueUniteLegale",
    "activitePrincipaleUniteLegale", "nomenclatureActivitePrincipaleUniteLegale",
    "activitePrincipaleNAF25UniteLegale",
    "economieSocialeSolidaireUniteLegale", "caractereEmployeurUniteLegale",
]
ul = load_sirene_ul(columns=UL_COLS)
print(ul.shape, f"{ul.memory_usage(deep=True).sum() / 1e9:.2f} Go |", mem())

(30148401, 15) 0.88 Go | process : 1.35 Go | disponible : 10.23 Go


In [3]:
ul.head()

,siren,statutDiffusionUniteLegale,unitePurgeeUniteLegale,dateCreationUniteLegale,dateDebut,etatAdministratifUniteLegale,trancheEffectifsUniteLegale,anneeEffectifsUniteLegale,categorieEntreprise,categorieJuridiqueUniteLegale,activitePrincipaleUniteLegale,nomenclatureActivitePrincipaleUniteLegale,activitePrincipaleNAF25UniteLegale,economieSocialeSolidaireUniteLegale,caractereEmployeurUniteLegale
0,325175,O,,2000-09-26,2018-02-07,A,NN,,PME,1000,32.12Z,NAFRev2,32.12Y,,
1,1807254,O,,1972-05-01,2014-12-31,C,NN,,,1000,85.59A,NAFRev2,,,
2,5410220,O,true,1954-12-25,1988-03-31,C,NN,,,1000,22.02,NAP,,,
3,5410345,O,true,,1984-12-25,C,NN,,,1000,79.06,NAP,,,
4,5410394,O,true,1954-12-25,1987-12-01,C,NN,,,1000,64.42,NAP,,,


### Observations

- Le fichier contient **30 148 401 entreprises**, une ligne par SIREN. On n'a que leur situation actuelle, pas leur historique.
- Avec les 15 colonnes choisies, le tableau prend **0,88 Go** de RAM, ça passe sans problème.

## 2. Valeurs manquantes

Petit piège : dans SIRENE, une case vide n'est pas un `NaN` mais une chaîne vide `""`. Il faut donc compter les deux.

In [4]:
manquants = pd.Series({c: (ul[c].isna() | (ul[c] == "")).mean() for c in ul.columns if c != "siren"})
manquants.sort_values(ascending=False).round(3)

caractereEmployeurUniteLegale                1.000
anneeEffectifsUniteLegale                    0.933
unitePurgeeUniteLegale                       0.859
economieSocialeSolidaireUniteLegale          0.778
categorieEntreprise                          0.638
activitePrincipaleNAF25UniteLegale           0.448
dateCreationUniteLegale                      0.039
dateDebut                                    0.013
nomenclatureActivitePrincipaleUniteLegale    0.001
activitePrincipaleUniteLegale                0.001
trancheEffectifsUniteLegale                  0.000
etatAdministratifUniteLegale                 0.000
statutDiffusionUniteLegale                   0.000
categorieJuridiqueUniteLegale                0.000
dtype: float64

### Observations

- `caractereEmployeurUniteLegale` est **complètement vide** (100 %). On ne pourra rien en faire.
- D'autres colonnes sont souvent vides : `anneeEffectifsUniteLegale` (93 %), `economieSocialeSolidaireUniteLegale` (78 %), `categorieEntreprise` (64 %) et le code NAF25 (45 %).
- `unitePurgeeUniteLegale` est vide à 86 %, mais là ce n'est pas un vrai manque : vide veut juste dire « pas purgée ».
- La date de création manque pour 3,9 % des entreprises. Le secteur, la forme juridique, l'état et la tranche d'effectifs sont presque toujours remplis (mais on verra en section 5 que la tranche vaut souvent `NN`, donc « inconnue »).

## 3. Diffusion et unités purgées

In [5]:
pd.crosstab(ul['statutDiffusionUniteLegale'], vide(ul['unitePurgeeUniteLegale'], 'non purgée'), margins=True)

unitePurgeeUniteLegale,non purgée,true,All
statutDiffusionUniteLegale,,,
O,22733057,4257162,26990219
P,3153915,4267,3158182
All,25886972,4261429,30148401


In [6]:
pd.crosstab(vide(ul['unitePurgeeUniteLegale'], 'non purgée'), ul['etatAdministratifUniteLegale'], margins=True)

etatAdministratifUniteLegale,A,C,All
unitePurgeeUniteLegale,,,
non purgée,16742414,9144558,25886972
true,0,4261429,4261429
All,16742414,13405987,30148401


### Observations

- **3 158 182 entreprises (10,5 %)** sont en diffusion partielle (`P`), donc avec des infos cachées.
- **4 261 429 entreprises (14 %)** sont purgées, et elles sont **toutes cessées**. C'est logique, l'Insee ne purge que des entreprises fermées.

## 4. État administratif : actives et cessées

In [7]:
ul['etatAdministratifUniteLegale'].value_counts()

etatAdministratifUniteLegale
A    16742414
C    13405987
Name: count, dtype: int64

In [8]:
# Dates en année (entier) : plus léger qu'une colonne datetime sur 30 M de lignes
ul["annee_creation"] = pd.to_numeric(ul["dateCreationUniteLegale"].astype("string").str[:4], errors="coerce").astype("Int16")
ul["annee_debut"] = pd.to_numeric(ul["dateDebut"].astype("string").str[:4], errors="coerce").astype("Int16")
ul[["annee_creation", "annee_debut"]].describe()

,annee_creation,annee_debut
count,28976552.0,29743878.0
mean,2006.239105,2013.728981
std,18.223381,12.548705
min,1.0,1.0
25%,1995.0,2008.0
50%,2011.0,2017.0
75%,2020.0,2023.0
max,2028.0,9202.0


In [9]:
# Unités cessées : année de cessation (= dateDebut de la dernière période)
cessees = ul["etatAdministratifUniteLegale"] == "C"
ul.loc[cessees, "annee_debut"].value_counts().sort_index().loc[2000:2026]

annee_debut
2000    173951
2001    176231
2002    174484
2003    173792
2004    181591
2005    182648
2006    176044
2007    126950
2008    262121
2009    186060
2010    216315
2011    244230
2012    257485
2013    263968
2014    291836
2015    293680
2016    487490
2017    367018
2018    378645
2019    410853
2020    500636
2021    562212
2022    529853
2023    718583
2024    932347
2025    969397
2026    438663
Name: count, dtype: Int64

In [10]:
# Créations par année (depuis 2000) et part encore active aujourd'hui
crea = (ul.loc[ul["annee_creation"] >= 2000]
          .groupby("annee_creation", observed=True)["etatAdministratifUniteLegale"]
          .agg(creations="size", part_active=lambda s: (s == "A").mean()))
crea.round(3)

,creations,part_active
annee_creation,,
2000,378236,0.543
2001,386405,0.548
2002,387848,0.560
2003,430201,0.568
2004,485692,0.567
2005,502184,0.577
2006,500686,0.592
2007,525828,0.588
2008,513951,0.590


In [11]:
# Dates aberrantes : création dans le futur ou très ancienne
print("création > 2026 :", (ul["annee_creation"] > 2026).sum())
print("création < 1900 :", (ul["annee_creation"] < 1900).sum())
print("création absente :", ul["annee_creation"].isna().sum())
print("dateDebut > 2026 :", (ul["annee_debut"] > 2026).sum())

création > 2026 : 1079
création < 1900 : 50
création absente : 1171849
dateDebut > 2026 : 1556


### Observations

- On a **16 742 414 entreprises actives** (56 %) et **13 405 987 cessées** (44 %).
- Pour les cessées, `dateDebut` nous donne l'année de fermeture. On passe d'environ 175 000 fermetures par an entre 2000 et 2006 à **932 347 en 2024** et **969 397 en 2025**. 2026 n'est pas finie (438 663 pour l'instant). Il y a aussi un pic à part en 2016 (487 490).
- Les créations augmentent aussi beaucoup : environ 380 000 par an en 2000, plus de 1,2 million par an depuis 2021.
- Plus l'entreprise est récente, plus elle a de chances d'être encore active : 54 % pour celles créées en 2000, 62 % pour 2020, 97 % pour 2026. ⚠️ Mais attention, c'est un peu normal : une entreprise créée l'an dernier a eu moins de temps pour fermer. On ne peut donc pas comparer les années directement.
- On trouve des **dates bizarres** : 1 079 créations après 2026 (jusqu'en 2028), 50 avant 1900 (dont une en l'an 1), 1 171 849 dates de création manquantes, et 1 556 `dateDebut` après 2026 (on voit même 8016 et 9202).

## 5. Taille : effectifs et catégorie d'entreprise

In [12]:
tranche = ul["trancheEffectifsUniteLegale"].astype("string")  # pour trier les codes dans l'ordre
pd.concat({"toutes": tranche.value_counts(),
           "part toutes": tranche.value_counts(normalize=True).round(4),
           "part actives": tranche[~cessees].value_counts(normalize=True).round(4)}, axis=1).sort_index()

,toutes,part toutes,part actives
trancheEffectifsUniteLegale,,,
00,116974,0.0039,0.0051
01,843594,0.028,0.0482
02,388639,0.0129,0.0226
03,198488,0.0066,0.0116
11,154640,0.0051,0.0091
12,94776,0.0031,0.0056
21,34495,0.0011,0.002
22,16189,0.0005,0.001
31,3144,0.0001,0.0002


In [13]:
ul['anneeEffectifsUniteLegale'].value_counts().sort_index().tail(10)

anneeEffectifsUniteLegale
        28141011
2024     2007390
Name: count, dtype: int64

In [14]:
pd.crosstab(vide(ul['categorieEntreprise'], '(vide)'), ul['etatAdministratifUniteLegale'], margins=True)

etatAdministratifUniteLegale,A,C,All
categorieEntreprise,,,
PME,8828993,1926473,10755466
(vide),7769705,11465256,19234961
ETI,103932,9655,113587
GE,39784,4603,44387
All,16742414,13405987,30148401


In [15]:
ul["caractereEmployeurUniteLegale"].value_counts(dropna=False)

caractereEmployeurUniteLegale
    30148401
Name: count, dtype: int64

### Observations

- La tranche d'effectifs vaut **`NN` (inconnue) pour 93,8 %** des entreprises, et pour 89,4 % des actives. On ne connaît donc le nombre de salariés que pour une petite partie.
- Quand on le connaît, c'est surtout `01` (1 à 2 salariés : 843 594 entreprises) et `02` (3 à 5 : 388 639). Seules 116 974 entreprises sont en `00` (0 salarié).
- Quand l'année des effectifs est remplie, c'est toujours **2024**. Ce sont donc les effectifs de 2024, pas ceux de l'année où l'entreprise a fait faillite.
- Pour la catégorie d'entreprise : 10,8 M de PME, 113 587 ETI, 44 387 GE, et **19,2 M de cases vides**, surtout chez les entreprises cessées (11,5 M).
- `caractereEmployeurUniteLegale` est bien vide partout.

## 6. Forme juridique

In [16]:
ul["cj_famille"] = ul["categorieJuridiqueUniteLegale"].astype("string").str[0].astype("category")
pd.crosstab(ul["cj_famille"], ul["etatAdministratifUniteLegale"], margins=True)

etatAdministratifUniteLegale,A,C,All
cj_famille,,,
1,7775857,9256870,17032727
2,231645,174467,406112
3,264887,91329,356216
4,1598,1548,3146
5,4333193,2998022,7331215
6,2535417,534697,3070114
7,93463,88701,182164
8,26147,8929,35076
9,1480207,251424,1731631


In [17]:
# Les 15 catégories juridiques les plus fréquentes parmi les unités actives
ul.loc[~cessees, "categorieJuridiqueUniteLegale"].value_counts().head(15)

categorieJuridiqueUniteLegale
1000    7775857
5499    2227940
6540    1904934
5710    1830812
9220    1246892
6599     284806
3220     238227
9110     166391
2110     157458
5202      97972
6598      81338
5599      66018
5485      60344
6597      48232
6541      45343
Name: count, dtype: int64

### Observations

- Les **entrepreneurs individuels** (famille `1`) sont les plus nombreux : 17,0 M, dont 7,8 M encore actifs.
- Les **sociétés commerciales** (famille `5`) : 7,3 M, dont 4,3 M actives. Chez les actives, on a surtout des SARL (`5499` : 2,2 M) et des SAS (`5710` : 1,8 M).
- Dans la famille `6`, ce sont surtout des **SCI** (`6540` : 1,9 M actives).
- Les **associations** (famille `9`) : 1,7 M, dont 1,2 M d'associations déclarées actives (`9220`).

## 7. Secteur d'activité (NAF)

In [18]:
pd.crosstab(vide(ul['nomenclatureActivitePrincipaleUniteLegale'], '(vide)'), ul['etatAdministratifUniteLegale'], margins=True)

etatAdministratifUniteLegale,A,C,All
nomenclatureActivitePrincipaleUniteLegale,,,
NAFRev2,15881659,8233148,24114807
NAP,46878,2308871,2355749
NAF1993,734211,1916310,2650521
NAFRev1,79666,926135,1005801
(vide),0,21523,21523
All,16742414,13405987,30148401


In [19]:
# Unités actives en NAFRev2 : les 20 divisions (2 premiers chiffres) les plus fréquentes
rev2 = ~cessees & (ul["nomenclatureActivitePrincipaleUniteLegale"] == "NAFRev2")
ul.loc[rev2, "activitePrincipaleUniteLegale"].astype("string").str[:2].value_counts().head(20)

activitePrincipaleUniteLegale
68    3674802
47    1221678
43     902419
01     759276
86     686511
70     649769
94     648069
96     554047
81     507354
85     467655
90     437931
56     429652
93     390614
46     332979
74     297794
62     268567
82     249959
53     242002
45     240265
66     232957
Name: count, dtype: int64[pyarrow]

In [20]:
# NAF25 : renseigné pour qui ?
naf25 = ul["activitePrincipaleNAF25UniteLegale"] != ""
pd.crosstab(ul["etatAdministratifUniteLegale"], naf25, colnames=["NAF25 renseigné"], margins=True)

NAF25 renseigné,False,True,All
etatAdministratifUniteLegale,,,
A,1514336,15228078,16742414
C,11985659,1420328,13405987
All,13499995,16648406,30148401


In [21]:
vide(ul['economieSocialeSolidaireUniteLegale'], '(vide)').value_counts()

economieSocialeSolidaireUniteLegale
(vide)    23459542
N          5271493
O          1417366
Name: count, dtype: int64

### Observations

- **24,1 M d'entreprises** ont un code dans la version actuelle (NAFRev2). Les anciennes versions (NAP, NAF1993, NAFRev1) concernent 6,0 M d'entreprises, surtout fermées. Mais il en reste quand même **734 211 actives en NAF1993**, 79 666 en NAFRev1 et 46 878 en NAP. 21 523 entreprises cessées n'ont aucun code.
- Chez les actives, les secteurs les plus fréquents sont `68` (immobilier : 3,7 M, ça colle avec le grand nombre de SCI), `47` (commerce de détail : 1,2 M), `43` (travaux de construction : 0,9 M), `01` (agriculture : 0,8 M) et `86` (santé : 0,7 M).
- Le code **NAF25** est rempli pour 91 % des entreprises actives (15,2 M), mais presque jamais pour les fermées (1,4 M).
- Économie sociale et solidaire : 1,4 M de « oui », 5,3 M de « non », et 23,5 M de cases vides.

## 8. Établissements sièges : localisation

In [22]:
SIEGES_COLS = ["siren", "nic", "codeCommuneEtablissement", "codePaysEtrangerEtablissement",
               "etatAdministratifEtablissement"]
sieges = load_sirene_sieges(columns=SIEGES_COLS)
print(sieges.shape, f"{sieges.memory_usage(deep=True).sum() / 1e9:.2f} Go |", mem())

(30126745, 5) 0.51 Go | process : 2.49 Go | disponible : 9.09 Go


In [23]:
# Un seul siège par SIREN ?
print("SIREN en double :", sieges["siren"].duplicated().sum())
print("UL sans siège dans le fichier :", (~ul["siren"].isin(sieges["siren"])).sum())

SIREN en double : 0


UL sans siège dans le fichier : 21656


In [24]:
# Département = 2 premiers caractères du code commune (3 pour l'outre-mer : 97x)
commune = sieges["codeCommuneEtablissement"].astype("string")
sieges["dep"] = commune.str[:2].where(~commune.str.startswith("97"), commune.str[:3]).astype("category")
print("siège à l'étranger :", (sieges["codePaysEtrangerEtablissement"] != "").sum())
print("code commune vide :", (commune.fillna("") == "").sum())
sieges["dep"].value_counts().head(15)

siège à l'étranger : 371980
code commune vide : 375276


dep
75    2449033
13    1068150
69     926863
59     870781
92     839479
33     801979
06     772029
93     763946
34     655420
31     627109
94     618565
83     616115
78     559652
38     541360
44     535130
Name: count, dtype: int64

In [25]:
# Cohérence état de l'UL / état du siège
etat = ul[["siren", "etatAdministratifUniteLegale"]].merge(
    sieges[["siren", "etatAdministratifEtablissement"]], on="siren", how="inner")
tab = pd.crosstab(etat["etatAdministratifUniteLegale"], etat["etatAdministratifEtablissement"], margins=True)
del etat
free_memory()
tab

etatAdministratifEtablissement,A,F,All
etatAdministratifUniteLegale,,,
A,15134835,1606454,16741289
C,20,13385436,13385456
All,15134855,14991890,30126745


### Observations

- On a 30 126 745 sièges, et **chaque entreprise en a un seul**. Par contre, **21 656 entreprises n'ont pas de siège** dans le fichier.
- 371 980 sièges sont **à l'étranger**, et 375 276 n'ont pas de code commune.
- Les départements avec le plus de sièges : Paris (2,4 M), les Bouches-du-Rhône (1,1 M), puis le Rhône, le Nord et les Hauts-de-Seine.
- Quand l'entreprise est cessée, son siège est presque toujours fermé (seulement 20 exceptions). Par contre, l'inverse n'est pas vrai : **1 606 454 entreprises actives ont un siège fermé**.

## 9. Les entreprises du BODACC sont-elles dans SIRENE ?

On reprend les SIREN des annonces BODACC de 2019 (on les extrait comme dans `bodacc.ipynb`) et on regarde si on les retrouve dans SIRENE.

In [26]:
bod = pd.read_csv("../data/raw/bodacc/procedures_collectives_2019.csv.gz", sep=";", dtype=str, usecols=["registre"])
siren_bodacc = pd.Series(bod["registre"].str.findall(r"\b\d{9}\b").explode().dropna().unique()).astype("int64")
del bod
trouves = siren_bodacc.isin(ul["siren"])
print(f"{trouves.sum()} SIREN BODACC 2019 trouvés dans SIRENE sur {len(siren_bodacc)} ({trouves.mean():.1%})")

121535 SIREN BODACC 2019 trouvés dans SIRENE sur 121579 (100.0%)


In [27]:
# État actuel dans SIRENE des entreprises passées au BODACC en 2019
ul_bod = ul.loc[ul["siren"].isin(siren_bodacc)]
display(ul_bod["etatAdministratifUniteLegale"].value_counts())
display(ul_bod["statutDiffusionUniteLegale"].value_counts())
ul_bod.loc[ul_bod["etatAdministratifUniteLegale"] == "C", "annee_debut"].value_counts().sort_index().loc[2015:2026]

etatAdministratifUniteLegale
C    87008
A    34527
Name: count, dtype: int64

statutDiffusionUniteLegale
O    120359
P      1176
Name: count, dtype: int64

annee_debut
2015      384
2016      691
2017     1598
2018     2773
2019    30850
2020    18006
2021    11905
2022     7055
2023     4720
2024     3693
2025     2710
2026     1361
Name: count, dtype: Int64

In [28]:
del ul_bod, siren_bodacc, trouves
print(free_memory())

process : 2.90 Go | disponible : 8.74 Go


### Observations

- On retrouve **presque tous les SIREN** du BODACC 2019 dans SIRENE : 121 535 sur 121 579. Il n'en manque que 44.
- Aujourd'hui, **87 008 de ces entreprises sont fermées** (72 %) et **34 527 sont encore actives** (28 %).
- Pour celles qui sont fermées, la fermeture date surtout de **2019 (30 850)** et de 2020 (18 006), puis ça s'étale jusqu'en 2026. Petit truc étonnant : 5 446 étaient déjà fermées avant 2019 (entre 2015 et 2018).
- 1 176 sont en diffusion partielle.

## 10. Petit point : c'est quoi un ratio financier ?

On va bientôt croiser SIRENE avec le fichier des **ratios financiers** (source BCE / INPI). Voici ce que c'est.

Chaque année, beaucoup d'entreprises doivent déposer leurs **comptes annuels** au greffe : le **bilan** (ce que l'entreprise possède et ce qu'elle doit, à la date de clôture) et le **compte de résultat** (ce qu'elle a gagné et dépensé sur l'année). Un **ratio financier**, c'est une division entre deux montants de ces comptes. Ça permet de comparer des entreprises de tailles différentes. Par exemple, une dette de 1 M€ n'a pas la même gravité pour une PME que pour un grand groupe, mais un taux d'endettement de 300 % est inquiétant dans les deux cas.

Dans notre fichier, une ligne = **une entreprise pour un exercice comptable** (une année de comptes), repérée par `siren` + `date_cloture_exercice`.

| Colonne | En clair |
|---|---|
| `chiffre_d_affaires` | Ce que l'entreprise a vendu sur l'année. |
| `marge_brute` | Les ventes moins le coût de ce qui a été acheté pour être revendu ou transformé. |
| `ebe` | Excédent brut d'exploitation : ce que l'activité rapporte avant les amortissements, les intérêts et les impôts. En gros, est-ce que le métier de base gagne de l'argent ? |
| `ebit` | Résultat d'exploitation : l'EBE moins l'usure du matériel (amortissements). |
| `resultat_net` | Le bénéfice (ou la perte) final, après tout. |
| `marge_ebe` | EBE / chiffre d'affaires : la part des ventes qui reste après les dépenses courantes. |
| `caf_sur_ca` | Capacité d'autofinancement / chiffre d'affaires : l'argent que l'entreprise génère elle-même, rapporté à ses ventes. |
| `resultat_courant_avant_impots_sur_ca` | Le résultat avant impôts et éléments exceptionnels, rapporté aux ventes. |
| `taux_d_endettement` | Dettes financières / capitaux propres. Plus c'est haut, plus l'entreprise dépend de ses prêteurs. |
| `autonomie_financiere` | Capitaux propres / total du bilan : la part de l'entreprise financée par ses propres fonds. |
| `capacite_de_remboursement` | Dettes financières / CAF : le nombre d'années qu'il faudrait pour rembourser les dettes avec l'argent généré. |
| `couverture_des_interets` | Est-ce que les bénéfices suffisent à payer les intérêts des emprunts ? |
| `ratio_de_liquidite` | Ce qui peut vite devenir du cash / les dettes à court terme. En dessous de 100 %, l'entreprise risque de ne pas pouvoir payer ses dettes du moment. |
| `ratio_de_vetuste` | À quel point le matériel est usé (amortissements / valeur d'achat). |
| `poids_bfr_exploitation_sur_ca` (et `_jours`) | Le besoin en fonds de roulement : l'argent bloqué dans les stocks et les factures pas encore payées par les clients, rapporté aux ventes (ou en nombre de jours de ventes). |
| `rotation_des_stocks_jours` | Combien de jours le stock reste en moyenne avant d'être vendu. |
| `credit_clients_jours` | En combien de jours les clients paient, en moyenne. |
| `credit_fournisseurs_jours` | En combien de jours l'entreprise paie ses fournisseurs, en moyenne. |
| `type_bilan` | `C` = bilan complet, `S` = bilan simplifié (petites entreprises), `K` = bilan consolidé (un groupe entier). |
| `confidentiality` | Si l'entreprise a demandé à garder une partie de ses comptes confidentielle. |

### Ce qu'on voit déjà dans le fichier des ratios

- 6 813 145 lignes pour **1 615 938 entreprises**, donc environ 4 exercices par entreprise.
- Les exercices vont surtout de **2016 à 2024** (environ 700 000 par an). 2025 est incomplète (405 531), et il y a quelques dates bizarres (1919, 2029).
- Bilans : 5 142 805 complets (`C`), 1 637 215 simplifiés (`S`), 33 125 consolidés (`K`).
- Confidentialité : 5 141 324 lignes publiques, 1 612 087 partiellement confidentielles, 57 045 en « RAPCAC » et 2 689 en publication simplifiée.
- **Presque toutes ces entreprises sont des sociétés commerciales** : 1 589 336 sur 1 615 938 (98 %), surtout des SAS (770 085) et des SARL (729 052). On ne trouve que 8 343 entrepreneurs individuels. C'est normal : un entrepreneur individuel n'a en général pas à publier ses comptes.
- On voit déjà des **valeurs énormes** : un chiffre d'affaires max de 2,9 × 10¹⁴ €, un ratio de liquidité max de 2,2 × 10⁹. Il faudra regarder ça de près.

## 11. Vérifications pour le nettoyage

Avant de décider comment nettoyer, on vérifie quelques points dans les données.

In [29]:
# On reprend les SIREN du BODACC 2019 et ceux du fichier des ratios pour voir l'impact de chaque choix
from data_utils import load_ratios

bod = pd.read_csv("../data/raw/bodacc/procedures_collectives_2019.csv.gz", sep=";", dtype=str, usecols=["registre"])
siren_bodacc = pd.Series(bod["registre"].str.findall(r"\b\d{9}\b").explode().dropna().unique()).astype("int64")
siren_ratios = pd.Series(load_ratios(columns=["siren"])["siren"].unique())
del bod
dans_bodacc = ul["siren"].isin(siren_bodacc)
dans_ratios = ul["siren"].isin(siren_ratios)
print(f"BODACC 2019 : {dans_bodacc.sum()} entreprises | avec ratios : {dans_ratios.sum()} | {free_memory()}")

BODACC 2019 : 121535 entreprises | avec ratios : 1615890 | process : 2.98 Go | disponible : 8.56 Go


### 11.1 Diffusion partielle : qu'est-ce qui est caché ?

In [30]:
# Taux de cases vides (ou masquées) selon le statut de diffusion
diffusion = ul["statutDiffusionUniteLegale"]
cols = [c for c in ul.columns if c not in ("siren", "statutDiffusionUniteLegale", "cj_famille", "annee_creation", "annee_debut")]
pd.DataFrame({s: {c: (ul.loc[diffusion == s, c].astype("string").fillna("").isin(["", "[ND]"])).mean() for c in cols}
              for s in ["O", "P"]}).round(3)

,O,P
unitePurgeeUniteLegale,0.842,0.999
dateCreationUniteLegale,0.043,0.000
dateDebut,0.015,0.000
etatAdministratifUniteLegale,0.000,0.000
trancheEffectifsUniteLegale,0.000,0.000
anneeEffectifsUniteLegale,0.926,0.996
categorieEntreprise,0.652,0.522
categorieJuridiqueUniteLegale,0.000,0.000
activitePrincipaleUniteLegale,0.001,0.000
nomenclatureActivitePrincipaleUniteLegale,0.001,0.000


In [31]:
# Les colonnes qu'on n'avait pas chargées : le sexe (UL) et le code postal (siège)
extra_ul = load_sirene_ul(columns=["siren", "statutDiffusionUniteLegale", "sexeUniteLegale"])
display(pd.crosstab(extra_ul["sexeUniteLegale"], extra_ul["statutDiffusionUniteLegale"]))
extra_sg = load_sirene_sieges(columns=["siren", "codePostalEtablissement"])
p_siren = extra_ul.loc[extra_ul["statutDiffusionUniteLegale"] == "P", "siren"]
cp_p = extra_sg.loc[extra_sg["siren"].isin(p_siren), "codePostalEtablissement"]
print("code postal masqué [ND] chez les P :", round((cp_p == "[ND]").mean(), 3))
del extra_ul, extra_sg, p_siren, cp_p
print(free_memory())

statutDiffusionUniteLegale,O,P
sexeUniteLegale,,
M,8730546,0
F,5163430,0
,13096243,0
[ND],0,3158182


code postal masqué [ND] chez les P : 1.0
process : 3.03 Go | disponible : 8.51 Go


In [32]:
# Quelles formes juridiques sont en diffusion partielle ?
ul.loc[diffusion == "P", "cj_famille"].value_counts()

cj_famille
1    3123644
2      19577
5       6735
3       4119
6       3758
9        302
7         41
8          6
4          0
Name: count, dtype: int64

### Observations

- Pour les entreprises en `P`, ce qui est caché c'est **le sexe** (toujours `[ND]`) et **le code postal du siège** (toujours `[ND]`). Par contre, **le code commune du siège reste là**, donc on a quand même le département.
- Tout le reste est rempli comme pour les autres (forme juridique, secteur, dates, état). En fait, les dates de création et le code NAF25 sont même plus souvent remplis chez les `P`.
- **99 % des entreprises en `P` sont des entrepreneurs individuels** (famille `1` : 3 123 644 sur 3 158 182). C'est logique : ce sont des personnes, qui peuvent demander à protéger leurs infos.

### 11.2 Taille de l'entreprise : qu'est-ce qu'on a à part les effectifs ?

In [33]:
nn = ul["trancheEffectifsUniteLegale"] == "NN"
cat_ok = ul["categorieEntreprise"] != ""
lignes = {}
for nom, m in [("toutes", pd.Series(True, index=ul.index)), ("BODACC 2019", dans_bodacc), ("avec ratios", dans_ratios)]:
    lignes[nom] = {
        "nb entreprises": m.sum(),
        "effectifs NN": (nn & m).sum() / m.sum(),
        "catégorie remplie": (cat_ok & m).sum() / m.sum(),
        "a des ratios": (dans_ratios & m).sum() / m.sum(),
        "ni effectifs, ni catégorie, ni ratios": (nn & ~cat_ok & ~dans_ratios & m).sum() / m.sum(),
    }
pd.DataFrame(lignes).T.round(3)

,nb entreprises,effectifs NN,catégorie remplie,a des ratios,"ni effectifs, ni catégorie, ni ratios"
toutes,30148401.0,0.938,0.362,0.054,0.626
BODACC 2019,121535.0,0.965,0.278,0.178,0.610
avec ratios,1615890.0,0.539,0.833,1.000,0.000


In [34]:
# Chez les entreprises du BODACC 2019 : catégorie d'entreprise selon que les effectifs sont connus ou non
pd.crosstab(vide(ul.loc[dans_bodacc, "categorieEntreprise"]), nn[dans_bodacc], colnames=["effectifs NN"])

effectifs NN,False,True
categorieEntreprise,,
PME,3979,29655
(vide),169,87519
ETI,123,54
GE,22,14


### Observations

- La **catégorie d'entreprise** (PME / ETI / GE) n'est remplie que pour 36 % des entreprises, et 28 % de celles du BODACC 2019. En plus, elle ne nous apprend pas grand-chose : quand elle est remplie au BODACC, c'est presque toujours « PME » (33 634 sur 33 847).
- Les **ratios** donnent une vraie mesure de la taille (le chiffre d'affaires est rempli sur toutes les lignes, et le type de bilan `S` signale les petites entreprises). Mais seulement 17,8 % des entreprises du BODACC 2019 ont des ratios.
- Au final, pour **61 % des entreprises du BODACC 2019**, on n'a **ni les effectifs, ni la catégorie, ni les ratios**. Pour elles, on n'a pas de mesure directe de la taille.

### 11.2 bis Et avec les établissements ?

Deux pistes : la **tranche d'effectifs de chaque établissement** (peut-être mieux remplie que celle de l'entreprise) et le **nombre d'établissements** de l'entreprise.

Le fichier `StockEtablissement` fait 10 Go décompressé. `load_sirene_etablissements()` le convertit une seule fois en parquet (environ 30 s), en gardant seulement 6 colonnes.

In [35]:
from data_utils import load_sirene_etablissements

etab = load_sirene_etablissements(columns=["siren", "trancheEffectifsEtablissement"])
print(etab.shape, "|", mem())
etab["trancheEffectifsEtablissement"].astype("string").value_counts(normalize=True).round(4).sort_index()

(44282364, 2) | process : 3.48 Go | disponible : 7.98 Go


trancheEffectifsEtablissement
00    0.0042
01     0.021
02    0.0111
03     0.006
11    0.0048
12     0.003
21    0.0011
22    0.0005
31    0.0001
32    0.0001
41    0.0001
42       0.0
51       0.0
52       0.0
53       0.0
NN    0.9479
Name: proportion, dtype: double[pyarrow]

In [36]:
# Une entreprise en NN a-t-elle au moins un établissement avec un effectif connu ?
etab_eff_connu = (etab["trancheEffectifsEtablissement"] != "NN").groupby(etab["siren"]).any()
nb_etab = etab.groupby("siren").size()
del etab
free_memory()

ul_eff_connu = ul["siren"].map(etab_eff_connu).fillna(False).astype(bool).to_numpy()
print("entreprises en NN avec au moins un établissement à effectif connu :", (nn.to_numpy() & ul_eff_connu).sum(), "sur", nn.sum())
print("  ... dont au BODACC 2019 :", (nn.to_numpy() & ul_eff_connu & dans_bodacc.to_numpy()).sum())
print("entreprises sans aucun établissement dans le fichier :", (~ul["siren"].isin(nb_etab.index)).sum())

entreprises en NN avec au moins un établissement à effectif connu : 26 sur 28285599
  ... dont au BODACC 2019 : 0


entreprises sans aucun établissement dans le fichier : 15963


In [37]:
# Nombre d'établissements par entreprise (6 = 6 ou plus)
nb_etab_ul = ul["siren"].map(nb_etab)
pd.DataFrame({
    "toutes": nb_etab_ul.clip(upper=6).value_counts().sort_index(),
    "BODACC 2019": nb_etab_ul[dans_bodacc].clip(upper=6).value_counts().sort_index(),
})

,toutes,BODACC 2019
siren,,
1.0,22179629,65991
2.0,5123847,32379
3.0,1670714,12641
4.0,631929,5244
5.0,260730,2373
6.0,265589,2907


In [38]:
display(nb_etab_ul.describe().round(2))
del nb_etab, nb_etab_ul, etab_eff_connu, ul_eff_connu
print(free_memory())

count    30132438.00
mean            1.47
std             5.80
min             1.00
25%             1.00
50%             1.00
75%             2.00
max         13131.00
Name: siren, dtype: float64

process : 3.07 Go | disponible : 8.59 Go


### Observations

- La tranche d'effectifs des établissements est encore **plus vide** que celle des entreprises : `NN` pour 94,8 % des 44,3 M d'établissements. Et quand elle est remplie, c'est aussi l'année 2024.
- Elle ne nous aide quasiment pas : seulement **26 entreprises** en `NN` ont un établissement avec un effectif connu, et **aucune** au BODACC 2019. On peut donc oublier cette piste.
- Le **nombre d'établissements**, lui, est connu pour presque toutes les entreprises (environ 16 000 n'ont aucun établissement dans le fichier). Une entreprise en a 1,47 en moyenne, et la plupart en ont un seul (22,2 M sur 30,1 M). Le max monte à 13 131.
- Au BODACC 2019, les entreprises ont un peu plus d'établissements : 54 % en ont un seul (65 991), 27 % en ont deux (32 379).
- ⚠️ Attention : ce nombre compte **tous les établissements que l'entreprise a eus**, même ceux fermés depuis longtemps. Ce n'est donc pas le nombre d'établissements au moment de la faillite. Pour l'avoir, il faudrait utiliser les dates d'ouverture et de fermeture des établissements.

### 11.3 Secteur d'activité : quelle nomenclature ?

In [39]:
nomenc = ul["nomenclatureActivitePrincipaleUniteLegale"]
display(pd.DataFrame({"BODACC 2019": nomenc[dans_bodacc].value_counts(), "avec ratios": nomenc[dans_ratios].value_counts()}))
naf25_ok = ul["activitePrincipaleNAF25UniteLegale"] != ""
print("NAF25 rempli chez les entreprises du BODACC 2019 :", round(naf25_ok[dans_bodacc].mean(), 3))

,BODACC 2019,avec ratios
nomenclatureActivitePrincipaleUniteLegale,,
NAFRev2,120191,1615608
NAFRev1,677,183
NAF1993,566,81
NAP,100,18
,1,0


NAF25 rempli chez les entreprises du BODACC 2019 : 0.535


### Observations

- Chez les entreprises du BODACC 2019, **98,9 % sont en NAFRev2** (120 191 sur 121 535). Chez celles qui ont des ratios, c'est quasiment 100 %.
- Le **NAF25** n'est rempli que pour 54 % des entreprises du BODACC 2019. C'est normal : il est surtout rempli pour les entreprises encore actives, et beaucoup d'entreprises en faillite sont fermées.

### 11.4 Les dates bizarres : est-ce qu'on peut les réparer ?

In [40]:
bizarre = lambda a: (a > 2026) | (a < 1900)
mauvaise_crea = bizarre(ul["annee_creation"]).fillna(False)
mauvais_debut = bizarre(ul["annee_debut"]).fillna(False)
print("création bizarre :", mauvaise_crea.sum(), "| dateDebut bizarre :", mauvais_debut.sum(), "| les deux :", (mauvaise_crea & mauvais_debut).sum())

# On compare avec les dates du siège, qui pourraient servir à réparer
dates_siege = load_sirene_sieges(columns=["siren", "dateCreationEtablissement", "dateDebut"]).rename(columns={"dateDebut": "dateDebutSiege"})
dates = ul.loc[mauvaise_crea | mauvais_debut, ["siren", "dateCreationUniteLegale", "dateDebut", "etatAdministratifUniteLegale"]].merge(dates_siege, on="siren", how="left")
del dates_siege
free_memory()
dates.sample(25, random_state=1)

création bizarre : 1129 | dateDebut bizarre : 1587 | les deux : 1101


,siren,dateCreationUniteLegale,dateDebut,etatAdministratifUniteLegale,dateCreationEtablissement,dateDebutSiege
108,104454640,2027-01-01,2027-01-01,A,2027-01-01,2027-01-01
1251,803291426,2014-05-15,2526-11-28,C,2014-05-15,2025-11-28
258,107185449,2027-01-01,2027-01-01,A,2027-01-01,2027-01-01
979,130592066,2027-09-24,2027-09-24,A,2027-09-24,2027-09-24
400,108297425,2027-01-01,2027-01-01,A,2027-01-01,2027-01-01
1048,130797277,2027-01-01,2027-01-01,A,2027-01-01,2027-01-01
1215,532540390,2011-07-01,2027-01-01,A,2027-01-01,2027-01-01
631,109332882,2027-01-01,2027-01-01,A,2027-01-01,2027-01-01
812,130077761,2027-01-01,2027-01-01,A,2027-01-01,2027-01-01
1223,750047870,2011-10-03,2107-10-16,C,2019-03-01,2021-08-31


In [41]:
annee = lambda s: pd.to_numeric(s.astype("string").str[:4], errors="coerce")
crea_bizarre = dates[bizarre(annee(dates["dateCreationUniteLegale"]))]
debut_bizarre = dates[bizarre(annee(dates["dateDebut"]))]
print("création bizarre, mais date de création du siège correcte :", annee(crea_bizarre["dateCreationEtablissement"]).between(1900, 2026).sum(), "sur", len(crea_bizarre))
print("dateDebut bizarre, mais dateDebut du siège correcte :", annee(debut_bizarre["dateDebutSiege"]).between(1900, 2026).sum(), "sur", len(debut_bizarre))
print("entreprises concernées dans le BODACC 2019 :", (dans_bodacc & (mauvaise_crea | mauvais_debut)).sum())
print("entreprises concernées dans les ratios :", (dans_ratios & (mauvaise_crea | mauvais_debut)).sum())

création bizarre, mais date de création du siège correcte : 21 sur 1129
dateDebut bizarre, mais dateDebut du siège correcte : 128 sur 1587
entreprises concernées dans le BODACC 2019 : 2


entreprises concernées dans les ratios : 32


### Observations

- On a 1 129 dates de création bizarres et 1 587 `dateDebut` bizarres. Souvent ce sont les mêmes entreprises (1 101 ont les deux).
- La plupart sont des **dates en 2027** (souvent le 01/01/2027), surtout pour des entrepreneurs individuels encore actifs. Ça ressemble à des entreprises déclarées en avance, avec une date de démarrage prévue. Ce n'est qu'une hypothèse, on ne l'a pas vérifiée.
- Certaines ont l'air d'être des **fautes de frappe** qu'on peut réparer avec la date du siège : `0004-09-24` au lieu de `2024-09-24`, `1190-08-21` au lieu de `1990-08-21`, `2526-11-28` au lieu de `2025-11-28`.
- Mais le siège ne donne une date correcte que dans peu de cas : pour 21 créations sur 1 129, et 128 `dateDebut` sur 1 587.
- Bonne nouvelle : ça ne touche presque pas les entreprises qui nous intéressent, **2 au BODACC 2019** et **32 dans les ratios**.

In [42]:
del siren_bodacc, siren_ratios, dans_bodacc, dans_ratios, dates, crea_bizarre, debut_bizarre
print(free_memory())

process : 3.26 Go | disponible : 8.21 Go


## Récapitulatif des décisions pour le nettoyage

| # | Sujet | Décision |
|---|---|---|
| 1 | **Quelles entreprises on garde** | **Toutes**. Il faut pouvoir relier n'importe quelle annonce du BODACC à son entreprise. On fera ensuite une analyse plus poussée sur celles qui ont des ratios. |
| 2 | **Entreprises fermées et purgées** | **Gardées**. |
| 3 | **Diffusion partielle (`P`)** | **Gardées**. On aurait supprimé ces lignes si des infos essentielles étaient cachées, mais ce n'est pas le cas : seuls le sexe et le code postal du siège sont masqués (section 11.1). Le code postal masqué (`[ND]`) passe en valeur manquante. |
| 4 | **Taille de l'entreprise** | On mesure la taille avec les **ratios** (chiffre d'affaires, type de bilan) quand l'entreprise en a. Sinon, tant pis : pour ces entreprises, on n'étudiera pas la défaillance selon la taille. Les effectifs de SIRENE (`NN` à 94 %), les effectifs des établissements (aucun apport) et le nombre d'établissements (compte aussi les établissements fermés) ne sont pas utilisés (sections 11.2 et 11.2 bis). |
| 5 | **Secteur d'activité** | On garde le code **NAFRev2**, qui couvre 98,9 % des entreprises du BODACC 2019 et presque 100 % de celles qui ont des ratios. On ne prend pas le NAF25, rempli pour seulement 54 % des entreprises du BODACC 2019. On travaille surtout avec la **section** (la lettre, 21 grands secteurs), et on garde aussi la **division** (2 chiffres, 88 secteurs) pour regarder plus en détail. Les codes dans une ancienne nomenclature (NAP, NAF1993, NAFRev1 : 1,1 % des entreprises du BODACC 2019) passent en valeur manquante. Le code d'attente `00.00Z` (pas encore de vrai code d'activité, 92 925 entreprises) aussi. |
| 6 | **Âge de l'entreprise** | Pour l'analyse avec les ratios : l'âge à la **date de clôture des comptes**. Pour les entreprises du BODACC sans ratios : l'âge à la **date du jugement**. |
| 7 | **D'où vient la défaillance** | **Uniquement du BODACC**. L'état « cessée » de SIRENE ne sert pas à définir la défaillance. |
| 8 | **Dates bizarres** | On répare l'année quand c'est possible (par exemple avec la date du siège). Sinon on la met en valeur manquante et **on garde la ligne**. On vérifie qu'on peut toujours analyser l'entreprise (section 11.4 : 2 entreprises au BODACC 2019, 32 dans les ratios). |
| 9 | **Âge négatif** (date de création SIRENE après la clôture des comptes) | L'âge passe en **valeur manquante**. Ce ne sont pas des demandes de création : ces entreprises ont des comptes jusqu'à 9 ans avant leur « création ». La date de création SIRENE correspond sans doute à un autre événement (reprise d'activité par exemple). |
